<a href="https://colab.research.google.com/github/Guido8383/AI-engineering-fundamentals/blob/main/GruppoC_Lezione6_GUIDO_RIGHI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI Engineering Fundamentals — Lezione 6
## Notebook Gruppo C

**ITS Novitas 4.0 | Martedì 09/06/2026 🏁**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [2]:
GRUPPO = "C"
MEMBRI = ["Guido Righi"]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

Gruppo C — Guido Righi


---
## 🎯 Tema del Gruppo C: Deploy — Dal Laptop a Internet

Guidate la classe nel processo di deploy su Streamlit Cloud.
Esplorate la gestione dei Secrets, il requirements.txt
e come costruire un portfolio GitHub professionale.

---
### Esercizio 1 — Preparare i file per il deploy *(guidato)*

Tre file sono necessari per deployare su Streamlit Cloud:
`app.py`, `requirements.txt` e il repository GitHub.
Createli tutti e verificate che siano corretti.

In [3]:
# Esercizio 1 — preparare i file per il deploy

# ── app.py ────────────────────────────────────────────────────────────
app_deploy = '''
import streamlit as st
import anthropic
import os

st.set_page_config(page_title="Chatbot WiData", page_icon="🤖")

# Gestione API key: Streamlit Cloud usa st.secrets
# In locale usa la variabile d'ambiente
if "ANTHROPIC_API_KEY" in st.secrets:
    os.environ["ANTHROPIC_API_KEY"] = st.secrets["ANTHROPIC_API_KEY"]

client = anthropic.Anthropic()


SYSTEM = """
Sei l'assistente virtuale di WiData Srl, una startup IoT di Sassari.
Il tuo compito è rispondere SOLO a domande riguardanti l'azienda, i sensori IoT, la tecnologia o la programmazione correlata.

REGOLA TASSATIVA: Se l'utente ti fa domande fuori da questo ambito (ad esempio cucina, ricette, sport, gossip o argomenti generali), devi rifiutarti in modo cortese ma fermo, dicendo che puoi parlare solo dei progetti e della tecnologia di WiData Srl.
"""
if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    st.title("⚙️ Impostazioni")
    temperature = st.slider("Temperature", 0.0, 1.0, 0.7, 0.1)
    if st.button("🗑️ Nuova chat"):
        st.session_state.messages = []
        st.rerun()

st.title("🤖 Chatbot WiData")

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Scrivi un messaggio..."):
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        risposta = ""
        placeholder = st.empty()
        with client.messages.stream(
            model="claude-haiku-4-5-20251001",
            max_tokens=500,
            temperature=temperature,
            system=SYSTEM,
            messages=st.session_state.messages
        ) as stream:
            for text in stream.text_stream:
                risposta += text
                placeholder.markdown(risposta + "▌")
        placeholder.markdown(risposta)

    st.session_state.messages.append({"role": "assistant", "content": risposta})
'''

# ── requirements.txt ──────────────────────────────────────────────────
requirements = """anthropic>=0.40.0
streamlit>=1.35.0
chromadb>=0.5.0
pypdf>=4.0.0
sentence-transformers>=3.0.0
requests>=2.31.0
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_deploy)
with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ File creati:")
print("  • app.py")
print("  • requirements.txt")
print()
print("Verificate che app.py gestisca correttamente la API key:")
print("  - In locale: os.environ['ANTHROPIC_API_KEY']")
print("  - Su Streamlit Cloud: st.secrets['ANTHROPIC_API_KEY']")
print()
print("Domande da rispondere:")
print("  1. Perché NON si usa .env su Streamlit Cloud?")
print("  2. Cosa succede se dimenticate di aggiungere il Secret prima del deploy?")
print("  3. Perché requirements.txt deve avere le versioni minime (>=)?")

✅ File creati:
  • app.py
  • requirements.txt

Verificate che app.py gestisca correttamente la API key:
  - In locale: os.environ['ANTHROPIC_API_KEY']
  - Su Streamlit Cloud: st.secrets['ANTHROPIC_API_KEY']

Domande da rispondere:
  1. Perché NON si usa .env su Streamlit Cloud?
  2. Cosa succede se dimenticate di aggiungere il Secret prima del deploy?
  3. Perché requirements.txt deve avere le versioni minime (>=)?




### 1. Perché NON si usa .env su Streamlit Cloud?

Non si usa il file `.env` per due motivi principali legati alla sicurezza e all'infrastruttura:

* **Sicurezza del Repository:** I file `.env` contengono dati sensibili (come le chiavi API) e per prassi non devono **mai** essere caricati su GitHub (vengono inseriti nel `.gitignore`). Poiché Streamlit Cloud esegue il deploy leggendo i file dal tuo repository GitHub, non troverebbe il file `.env`.
* **Infrastruttura nativa:** Streamlit Cloud dispone di un proprio sistema crittografato e sicuro per la gestione delle chiavi chiamato **Secrets Management**. Invece di leggere variabili d'ambiente da un file di testo, Streamlit inietta questi segreti direttamente in un dizionario speciale richiamabile nel codice tramite `st.secrets`.

### 2. Cosa succede se dimenticate di aggiungere il Secret prima del deploy?

L'applicazione si avvierà e l'interfaccia si caricherà, ma andrà **in crash (errore) non appena proverà a eseguire una chiamata all'API**.
Nello specifico del vostro codice, otterrete molto probabilmente un `KeyError` se il codice cerca `st.secrets["ANTHROPIC_API_KEY"]` e non lo trova, oppure un `AuthenticationError` da parte del client di Anthropic se l'API key risulta vuota o invalida. L'app non riuscirà a generare la risposta del chatbot.

### 3. Perché requirements.txt deve avere le versioni minime (>=)?

Usare le versioni minime (`>=`) offre il giusto equilibrio tra stabilità e flessibilità:

* **Previene rotture:** Assicura che l'ambiente installi una versione della libreria che contiene tutte le funzionalità (o i bug fix) necessarie affinché il tuo codice funzioni (es. se una funzione specifica di Streamlit è stata introdotta nella `1.35.0`).
* **Flessibilità di risoluzione:** A differenza del blocco rigido della versione (es. `==1.35.0`), usare `>=` permette all'installer (pip) di trovare versioni compatibili se ci sono dipendenze incrociate o conflitti tra diverse librerie. Permette inoltre all'app di beneficiare di aggiornamenti di sicurezza minori senza dover modificare manualmente il file.


---
### Esercizio 2 — Simulare i Secrets in locale *(guidato)*

Su Streamlit Cloud i Secrets si configurano nel pannello.
In locale potete simularli con un file `.streamlit/secrets.toml`.
Costruite entrambe le versioni e verificate che funzionino.

In [4]:
# Esercizio 2 — simulare i Secrets in locale

import os

# Crea la directory .streamlit se non esiste
os.makedirs(".streamlit", exist_ok=True)

# Il file secrets.toml simula i Secrets di Streamlit Cloud in locale
# ATTENZIONE: questo file NON va su GitHub — va nel .gitignore!
api_key = os.environ.get("ANTHROPIC_API_KEY", "sk-ant-PLACEHOLDER")

secrets_toml = f'ANTHROPIC_API_KEY = "{api_key}"\n'
with open(".streamlit/secrets.toml", "w") as f:
    f.write(secrets_toml)

print("✅ .streamlit/secrets.toml creato")
print()

# Crea il .gitignore corretto
gitignore = """.streamlit/secrets.toml
.env
__pycache__/
*.pyc
.DS_Store
chroma_db/
*.json
"""
with open(".gitignore", "w") as f:
    f.write(gitignore)

print("✅ .gitignore creato")
print()
print("File da NON committare su GitHub:")
print("  ❌ .streamlit/secrets.toml — contiene la API key")
print("  ❌ .env — contiene la API key")
print("  ❌ chroma_db/ — database locale")
print()
print("File da committare su GitHub:")
print("  ✅ app.py")
print("  ✅ requirements.txt")
print("  ✅ .gitignore")
print("  ✅ README.md")
print()

# Verifica che il pattern if/else funzioni
print("Test pattern API key:")
print("  Su Streamlit Cloud: st.secrets['ANTHROPIC_API_KEY']")
print("  In locale (.streamlit/secrets.toml): stessa sintassi, file locale")
print("  In locale (.env): os.environ.get('ANTHROPIC_API_KEY')")

✅ .streamlit/secrets.toml creato

✅ .gitignore creato

File da NON committare su GitHub:
  ❌ .streamlit/secrets.toml — contiene la API key
  ❌ .env — contiene la API key
  ❌ chroma_db/ — database locale

File da committare su GitHub:
  ✅ app.py
  ✅ requirements.txt
  ✅ .gitignore
  ✅ README.md

Test pattern API key:
  Su Streamlit Cloud: st.secrets['ANTHROPIC_API_KEY']
  In locale (.streamlit/secrets.toml): stessa sintassi, file locale
  In locale (.env): os.environ.get('ANTHROPIC_API_KEY')


---
### Esercizio 3 — Il README professionale *(libero)*

Il README è il biglietto da visita del repository.
Costruite un README professionale per il chatbot WiData
che includa tutti gli elementi necessari per impressionare
un datore di lavoro o un cliente.

In [5]:
# Esercizio 3 — README professionale

# Struttura del README — completate ogni sezione
readme = """
# 🤖 Chatbot WiData

<!-- Badge dello stack tecnologico -->
![Python](https://img.shields.io/badge/Python-3.11-blue)
![Anthropic](https://img.shields.io/badge/Anthropic-Claude-orange)
![Streamlit](https://img.shields.io/badge/Streamlit-1.35-red)

## 📋 Descrizione

<!-- TODO: 2-3 frasi che spiegano cosa fa il chatbot -->
...

## 🚀 Demo

<!-- TODO: link a Streamlit Cloud quando deployato -->
**Live**: [chatbot-widata.streamlit.app](https://...)

## ✨ Funzionalità

<!-- TODO: lista delle features principali -->
- ...

## 🛠 Stack tecnologico

<!-- TODO: tecnologie usate con descrizione breve -->
| Tecnologia | Uso |
|------------|-----|
| Claude Haiku | ... |
| ChromaDB | ... |
| Streamlit | ... |

## 📐 Architettura

<!-- TODO: descrizione del flusso RAG + tool in 3-4 righe -->
...

## ⚙️ Esecuzione in locale

```bash
# Clone
git clone https://github.com/TUO_USERNAME/AI-engineering-fundamentals
cd AI-engineering-fundamentals

# Installa dipendenze
pip install -r requirements.txt

# Configura API key
mkdir .streamlit
echo 'ANTHROPIC_API_KEY = "sk-ant-..."' > .streamlit/secrets.toml

# Avvia
streamlit run app.py
```

## 📍 Posizionamento Crawl-Walk-Run

<!-- TODO: dove si posiziona il chatbot e perché -->
Il chatbot si posiziona in zona **WALK** perché...

## 🔮 Passo successivo

<!-- TODO: cosa fareste per portarlo a RUN -->
Per avanzare verso RUN implementeremmo...

---
*Progetto realizzato durante il corso AI Engineering Fundamentals*
*ITS Novitas 4.0 — Sassari, 2026*
"""

with open("README_template.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("✅ README_template.md creato")
print()
print("TODO: completate ogni sezione del README.")
print("Poi rinominatelo README.md e committatelo su GitHub.")
print()
print("Elementi essenziali per un README professionale:")
print("  1. Descrizione chiara in 2-3 frasi")
print("  2. Link al deploy pubblico in evidenza")
print("  3. Screenshot o GIF dell'interfaccia")
print("  4. Stack tecnologico con badge")
print("  5. Istruzioni per eseguire in locale")
print("  6. Posizionamento Crawl-Walk-Run")

✅ README_template.md creato

TODO: completate ogni sezione del README.
Poi rinominatelo README.md e committatelo su GitHub.

Elementi essenziali per un README professionale:
  1. Descrizione chiara in 2-3 frasi
  2. Link al deploy pubblico in evidenza
  3. Screenshot o GIF dell'interfaccia
  4. Stack tecnologico con badge
  5. Istruzioni per eseguire in locale
  6. Posizionamento Crawl-Walk-Run


---
### Esercizio 4 — Guida al deploy live *(libero)*

Preparate una guida passo per passo per deployare su Streamlit Cloud.
Testatela voi stessi — alla fine dovete avere un URL pubblico.
Poi usate la guida per aiutare i compagni durante la lezione.

In [6]:
# Esercizio 4 — guida al deploy e checklist

print("=" * 60)
print("GUIDA DEPLOY SU STREAMLIT CLOUD")
print("=" * 60)
print()
print("PREREQUISITI")
print("  □ Account GitHub con repository pubblica")
print("  □ app.py nella root o in una sottocartella")
print("  □ requirements.txt nella root del repository")
print("  □ Account su share.streamlit.io (gratuito)")
print()
print("PASSI")
print("  1. Push su GitHub")
print("     git add app.py requirements.txt .gitignore README.md")
print('     git commit -m "Deploy chatbot WiData"')
print("     git push")
print()
print("  2. Vai su share.streamlit.io")
print("     → New app")
print("     → Repository: tuo-username/AI-engineering-fundamentals")
print("     → Branch: main")
print("     → Main file path: lezione6/app.py")
print()
print("  3. Advanced settings → Secrets")
print('     ANTHROPIC_API_KEY = "sk-ant-..."')
print("     ⚠️ AGGIUNGERE IL SECRET PRIMA DI CLICCARE DEPLOY")
print()
print("  4. Clicca Deploy")
print("     Attendi 3-5 minuti (prima build lenta: scarica sentence-transformers)")
print()
print("  5. URL pubblico")
print("     https://TUONOME-chatbot-widata-HASH.streamlit.app")
print()
print("ERRORI COMUNI")
print("  ❌ ModuleNotFoundError → requirements.txt incompleto")
print("  ❌ AuthenticationError → Secret non configurato")
print("  ❌ File non trovato → path sbagliato nel deploy")
print("  ❌ Build lenta → normale, sentence-transformers è ~800MB")
print()

# Compilate qui dopo il deploy
MIO_URL = ""  # ← incollate il vostro URL
MIO_GITHUB = ""  # ← incollate il link alla repo

if MIO_URL:
    print(f"🚀 Il vostro chatbot è live: {MIO_URL}")
    print(f"📁 Repository: {MIO_GITHUB}")
else:
    print("⚠️  Completate il deploy e incollate l'URL sopra!")

GUIDA DEPLOY SU STREAMLIT CLOUD

PREREQUISITI
  □ Account GitHub con repository pubblica
  □ app.py nella root o in una sottocartella
  □ requirements.txt nella root del repository
  □ Account su share.streamlit.io (gratuito)

PASSI
  1. Push su GitHub
     git add app.py requirements.txt .gitignore README.md
     git commit -m "Deploy chatbot WiData"
     git push

  2. Vai su share.streamlit.io
     → New app
     → Repository: tuo-username/AI-engineering-fundamentals
     → Branch: main
     → Main file path: lezione6/app.py

  3. Advanced settings → Secrets
     ANTHROPIC_API_KEY = "sk-ant-..."
     ⚠️ AGGIUNGERE IL SECRET PRIMA DI CLICCARE DEPLOY

  4. Clicca Deploy
     Attendi 3-5 minuti (prima build lenta: scarica sentence-transformers)

  5. URL pubblico
     https://TUONOME-chatbot-widata-HASH.streamlit.app

ERRORI COMUNI
  ❌ ModuleNotFoundError → requirements.txt incompleto
  ❌ AuthenticationError → Secret non configurato
  ❌ File non trovato → path sbagliato nel deploy
  ❌ 

---
## 📊 Preparate la presentazione (5 slide)

1. **I 3 file necessari** — app.py, requirements.txt, README.md
2. **La gestione dei Secrets** — perché non .env, come si configura
3. **Il deploy passo per passo** — mostrate i passaggi su Streamlit Cloud
4. **Il README professionale** — mostrate il vostro con tutti gli elementi
5. **Il vostro URL live** — demo del chatbot deployato

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*